# Multimodal Cancer Classification Challenge 2026 — v18-all

**Strategy update after v18-floor.** v18-floor (all toggles OFF, single full-data model) scored **LB 0.5529** — much lower than the predicted 0.64. The 0.078 drop from v17 (0.6311) is too large to be just "lost the ensemble"; either the ensemble or the three improvements carried much more lift than I estimated.

**v18-all (this commit)**: same single full-data model, but with **all three improvements ON**. This isolates the question: was the v17 lift from the improvements or from the ensemble?

## v18 commits to date and plan

| Commit | Toggles | Ensemble | LB | Verdict |
|---|---|---|---|---|
| v18-floor | all OFF | none | **0.5529** | floor established (lower than expected) |
| **v18-all (this commit)** | **all ON** | **none** | **TBD** | **separates ensemble vs improvements** |
| v18.1 | only stain norm | none | TBD | only if v18-all ≥ 0.62 |
| v18.2 | only AdaBN | none | TBD | only if v18-all ≥ 0.62 |
| v18.3 | only multi-scale TTA | none | TBD | only if v18-all ≥ 0.62 |
| v18.5 (final) | winners + 3-fold CV × 2 seeds | yes | TBD | last commit before deadline |

## Reading the v18-all result

- **v18-all ≥ 0.62**: the three improvements collectively explain the v17 lift. Continue ablating one at a time to find which is the main driver, then ensemble.
- **v18-all in 0.58–0.62**: improvements and ensemble both contributed roughly equally. Both keep, both ablate.
- **v18-all < 0.58**: ensemble was carrying the lift; improvements are noise/negative. Skip to ensemble-restore commit immediately.

## Compute budget on T4

| Stage | Time |
|---|---|
| JPEG cache (one-time) | ~70 min |
| Compute test pixel stats | ~30 s |
| Train 1 full-data model (8 epochs, no val) | ~24 min |
| 24-way TTA + AdaBN, 1 ckpt | ~12 min |
| **Total** | **~1h 47min** |

(Cache is the dominant cost; the actual training-and-inference work is ~36 min. AdaBN + 24-way TTA tripples inference time vs v18-floor's 8-way TTA, ~12 min instead of ~4 min.)

## IMPORTANT — DO NOT JUST CLICK RUN ALL

To get a `submission.csv` you must:

1. Click **Save Version** (top-right green button)
2. Choose **Save & Run All (Commit)**
3. Add a description like "v18-all: 3 improvements ON, single model"
4. **Wait for the commit to finish** (you'll get a notification)
5. Open the saved version → Output tab → submission.csv is there

Running cells one-by-one in editor mode trains the model but does NOT persist outputs as a Saved Version.

In [ ]:
import os
os.environ["PYTHONUNBUFFERED"] = "1"

import re, io, json, time, random, glob, functools, gc
from pathlib import Path

print = functools.partial(print, flush=True)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision import models
from sklearn.metrics import roc_auc_score
from PIL import Image
import matplotlib.pyplot as plt

print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(),
      "n_gpu:", torch.cuda.device_count())
!nvidia-smi -L

In [ ]:
DATA_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/rafaelproena/a3-adl"),
    Path("/kaggle/input/a3-adl"),
    Path("/kaggle/input/competitions/multimodal-cancer-classification-challenge-2026"),
]
DATA_ROOT = next((p for p in DATA_ROOT_CANDIDATES if (p / "train.csv").exists()), None)
assert DATA_ROOT is not None, f"train.csv not found at any of {DATA_ROOT_CANDIDATES}"
print("DATA_ROOT =", DATA_ROOT)

OUT_DIR = Path("/kaggle/working/runs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# === ABLATION TOGGLES — v18-all (all THREE ON, still single model) ===
# v18-floor (all OFF) scored LB 0.5529 — much lower than expected.
# v17 (all ON + 7-model ensemble) scored LB 0.6311.
# This commit isolates: do the 3 improvements collectively carry the lift, or did the ensemble?
USE_TEST_STAIN_NORM = True   # test BF/FL pixel stats
USE_ADABN           = True   # update BN running stats on test before inference
USE_MULTISCALE_TTA  = True   # 24-way TTA (D4 × 3 scales) instead of 8-way D4
TTA_SCALES          = (112, 128, 144)

# === Optional ablations (default OFF) ===
LABEL_SMOOTHING     = 0.0    # set to 0.05 to enable
USE_EFFICIENTNET    = False  # set to True to swap ResNet-18 -> EfficientNet-B0 (timm)

# === Training (v11 / v17 settings) ===
BASE_SEED   = 1
EPOCHS      = 8           # v17 median best_ep was 7 -> +1 for full-data extra capacity
BATCH_SIZE  = 128
LR          = 3e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP   = 1.0
MIXUP_ALPHA = 0.1
DROPOUT     = 0.3

NUM_WORKERS = 2
PATIENTS_PER_BATCH = 4

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    torch.cuda.set_device(0)

# v11 hardcoded stats (used when USE_TEST_STAIN_NORM=False)
BF_MEAN, BF_STD = 0.504, 0.216
FL_MEAN, FL_STD = 0.100, 0.144

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

seed_everything(BASE_SEED)

print(f"\nConfig (v18-all — all 3 improvements ON, single model):")
print(f"  USE_TEST_STAIN_NORM = {USE_TEST_STAIN_NORM}")
print(f"  USE_ADABN           = {USE_ADABN}")
print(f"  USE_MULTISCALE_TTA  = {USE_MULTISCALE_TTA}  scales={TTA_SCALES}")
print(f"  LABEL_SMOOTHING     = {LABEL_SMOOTHING}")
print(f"  USE_EFFICIENTNET    = {USE_EFFICIENTNET}")

In [ ]:
PAT_RE = re.compile(r"^pat_(\d+)_image_\d+\.jpg$")

def parse_patient_id(filename):
    m = PAT_RE.match(Path(filename).name)
    return int(m.group(1)) if m else None

def load_train_df(path):
    df = pd.read_csv(path); df.columns = [c.strip() for c in df.columns]
    df["patient_id"] = df["Name"].map(parse_patient_id).astype(int)
    return df

def load_test_df(path):
    df = pd.read_csv(path); df.columns = [c.strip() for c in df.columns]
    return df

def cache_split(names, bf_dir, fl_dir, label=""):
    bf_dir, fl_dir = Path(bf_dir), Path(fl_dir)
    bf, fl = {}, {}
    t0 = time.time()
    for i, n in enumerate(names):
        with open(bf_dir / n, "rb") as f: bf[n] = f.read()
        with open(fl_dir / n, "rb") as f: fl[n] = f.read()
        if (i + 1) % 20000 == 0:
            print(f"  [{label}] cached {i+1}/{len(names)} in {time.time()-t0:.1f}s")
    print(f"  [{label}] cached {len(names)} in {time.time()-t0:.1f}s")
    return bf, fl

class CachedCellDataset(Dataset):
    def __init__(self, df, bf_cache, fl_cache, bf_tf, fl_tf, paired_tf=None):
        self.df = df.reset_index(drop=True)
        self.bf_cache = bf_cache; self.fl_cache = fl_cache
        self.bf_tf = bf_tf; self.fl_tf = fl_tf
        self.paired_tf = paired_tf
    def __len__(self): return len(self.df)
    @staticmethod
    def _decode(buf): return Image.open(io.BytesIO(buf)).convert("L")
    def __getitem__(self, idx):
        row = self.df.iloc[idx]; name = row["Name"]
        bf = self.bf_tf(self._decode(self.bf_cache[name]))
        fl = self.fl_tf(self._decode(self.fl_cache[name]))
        if self.paired_tf is not None:
            bf, fl = self.paired_tf(bf, fl)
        label = int(row["Diagnosis"]) if "Diagnosis" in row else -1
        return {"bf": bf, "fl": fl, "label": label, "name": name}

class PatientBalancedSampler(Sampler):
    def __init__(self, df, batch_size, patients_per_batch=4, seed=0):
        assert batch_size % patients_per_batch == 0
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.per_pat = batch_size // patients_per_batch
        self.patients_per_batch = patients_per_batch
        self.rng = np.random.default_rng(seed)
        self.by_pat = {p: np.array(g.index.tolist())
                       for p, g in self.df.groupby("patient_id")}
        self.patients = list(self.by_pat.keys())
        self.epoch_len = len(self.df) // batch_size * batch_size
    def __len__(self): return self.epoch_len
    def __iter__(self):
        out = []
        for _ in range(self.epoch_len // self.batch_size):
            pats = self.rng.choice(self.patients,
                                   size=min(self.patients_per_batch, len(self.patients)),
                                   replace=False)
            for p in pats:
                idxs = self.by_pat[p]
                out.extend(self.rng.choice(idxs, size=self.per_pat,
                                           replace=len(idxs) < self.per_pat).tolist())
        return iter(out)

In [ ]:
def _make_resnet18_branch(pretrained=True):
    weights = "DEFAULT" if pretrained else None
    net = models.resnet18(weights=weights)
    w = net.conv1.weight.data
    new_conv = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    if pretrained:
        new_conv.weight.data = w.mean(dim=1, keepdim=True)
    net.conv1 = new_conv
    fd = net.fc.in_features; net.fc = nn.Identity()
    return net, fd

def _make_effnet_b0_branch(pretrained=True):
    import timm
    net = timm.create_model("efficientnet_b0", pretrained=pretrained,
                            num_classes=0, global_pool="avg")
    old = net.conv_stem
    new_conv = nn.Conv2d(1, old.out_channels, kernel_size=old.kernel_size,
                        stride=old.stride, padding=old.padding, bias=False)
    if pretrained:
        new_conv.weight.data = old.weight.data.mean(dim=1, keepdim=True)
    net.conv_stem = new_conv
    return net, net.num_features  # 1280

def _make_branch(pretrained=True):
    if USE_EFFICIENTNET:
        return _make_effnet_b0_branch(pretrained)
    return _make_resnet18_branch(pretrained)

class MultimodalClassifier(nn.Module):
    def __init__(self, pretrained=True, dropout=DROPOUT):
        super().__init__()
        self.bf_branch, fd = _make_branch(pretrained)
        self.fl_branch, _  = _make_branch(pretrained)
        hidden = 512 if fd >= 512 else 256
        self.head = nn.Sequential(
            nn.Linear(fd * 2, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )
    def forward(self, bf, fl):
        feat = torch.cat([self.bf_branch(bf), self.fl_branch(fl)], dim=1)
        return self.head(feat).squeeze(-1)

with torch.no_grad():
    _m = MultimodalClassifier(pretrained=False).cpu()
    _x = torch.zeros(2, 1, 128, 128)
    n_params = sum(p.numel() for p in _m.parameters())
    print(f"Output shape: {_m(_x, _x).shape}   params: {n_params / 1e6:.1f}M")
    print(f"Backbone: {'EfficientNet-B0' if USE_EFFICIENTNET else 'ResNet-18'}")
    del _m, _x

In [ ]:
df_train = load_train_df(DATA_ROOT / "train.csv")
df_test  = load_test_df(DATA_ROOT / "sampleSubmission.csv")
print(f"Train: {len(df_train)} cells, {df_train['patient_id'].nunique()} patients, "
      f"pos rate {df_train['Diagnosis'].mean():.4f}")
print(f"Test:  {len(df_test)} cells")

print("\n--- Caching JPEG bytes into RAM ---")
bf_train_cache, fl_train_cache = cache_split(
    df_train["Name"].tolist(),
    DATA_ROOT / "BF" / "train", DATA_ROOT / "FL" / "train", label="train")
bf_test_cache, fl_test_cache = cache_split(
    df_test["Name"].tolist(),
    DATA_ROOT / "BF" / "test", DATA_ROOT / "FL" / "test", label="test")
approx_mb = (sum(len(b) for b in bf_train_cache.values()) +
             sum(len(b) for b in fl_train_cache.values()) +
             sum(len(b) for b in bf_test_cache.values()) +
             sum(len(b) for b in fl_test_cache.values())) / (1024 * 1024)
print(f"\nApprox RAM used by JPEG cache: {approx_mb:.0f} MB")

# Compute pixel statistics for stain normalization.
def _sample_pixel_stats(cache, names, n_sample=1500):
    rng = np.random.default_rng(42)
    sampled = rng.choice(names, size=min(n_sample, len(names)), replace=False)
    pixels = []
    for n in sampled:
        img = np.asarray(Image.open(io.BytesIO(cache[n])).convert("L"),
                         dtype=np.float32) / 255.0
        pixels.append(img.ravel())
    pixels = np.concatenate(pixels)
    return float(pixels.mean()), float(pixels.std())

if USE_TEST_STAIN_NORM:
    BF_MEAN_T, BF_STD_T = _sample_pixel_stats(bf_test_cache, df_test["Name"].tolist())
    FL_MEAN_T, FL_STD_T = _sample_pixel_stats(fl_test_cache, df_test["Name"].tolist())
    BF_MEAN_R, BF_STD_R = _sample_pixel_stats(bf_train_cache, df_train["Name"].tolist())
    FL_MEAN_R, FL_STD_R = _sample_pixel_stats(fl_train_cache, df_train["Name"].tolist())
    print(f"\nPixel statistics:")
    print(f"  BF train: mean={BF_MEAN_R:.4f} std={BF_STD_R:.4f}")
    print(f"  BF test:  mean={BF_MEAN_T:.4f} std={BF_STD_T:.4f}")
    print(f"  FL train: mean={FL_MEAN_R:.4f} std={FL_STD_R:.4f}")
    print(f"  FL test:  mean={FL_MEAN_T:.4f} std={FL_STD_T:.4f}")
    BF_MEAN, BF_STD = BF_MEAN_T, BF_STD_T
    FL_MEAN, FL_STD = FL_MEAN_T, FL_STD_T
    print(f"  -> using TEST stats (USE_TEST_STAIN_NORM=True)")
else:
    print(f"\nUsing v11 hardcoded normalization (USE_TEST_STAIN_NORM=False)")

In [ ]:
def _to_tensor_norm(mean, std):
    def fn(img):
        t = TF.to_tensor(img)
        return TF.normalize(t, [mean], [std])
    return fn

to_tensor_bf = _to_tensor_norm(BF_MEAN, BF_STD)
to_tensor_fl = _to_tensor_norm(FL_MEAN, FL_STD)

class PairedGeoAug:
    def __init__(self, p_hflip=0.5, p_vflip=0.5, rot90=True, max_rot=10.0):
        self.p_hflip = p_hflip; self.p_vflip = p_vflip
        self.rot90 = rot90; self.max_rot = max_rot
    def __call__(self, bf, fl):
        if self.rot90:
            k = random.randint(0, 3)
            if k:
                bf = torch.rot90(bf, k, dims=(-2, -1))
                fl = torch.rot90(fl, k, dims=(-2, -1))
        if random.random() < self.p_hflip: bf, fl = TF.hflip(bf), TF.hflip(fl)
        if random.random() < self.p_vflip: bf, fl = TF.vflip(bf), TF.vflip(fl)
        if self.max_rot > 0:
            a = random.uniform(-self.max_rot, self.max_rot)
            bf, fl = TF.rotate(bf, a), TF.rotate(fl, a)
        return bf, fl

def train_modality_transform(modality):
    norm = to_tensor_bf if modality == "bf" else to_tensor_fl
    return T.Compose([T.ColorJitter(brightness=0.2, contrast=0.2), norm])

def eval_modality_transform(modality):
    return to_tensor_bf if modality == "bf" else to_tensor_fl

In [ ]:
def mixup_batch(bf, fl, y, alpha=0.1):
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(bf.size(0), device=bf.device)
    return (lam * bf + (1 - lam) * bf[idx],
            lam * fl + (1 - lam) * fl[idx],
            lam * y  + (1 - lam) * y[idx])

def smooth(y, eps):
    """Label smoothing: 0/1 -> eps/2 / (1 - eps/2). Only applied if eps > 0."""
    if eps <= 0: return y
    return y * (1.0 - eps) + eps * 0.5

def run_epoch_train(model, loader, optimizer, scaler, criterion, sched, log_every=200):
    model.train()
    losses, hard_ys, ps = [], [], []
    t_last = time.time()
    for i, batch in enumerate(loader):
        bf = batch["bf"].to(DEVICE, non_blocking=True)
        fl = batch["fl"].to(DEVICE, non_blocking=True)
        y  = batch["label"].float().to(DEVICE, non_blocking=True)
        hard_ys.append(batch["label"].numpy())
        if MIXUP_ALPHA > 0:
            bf, fl, y = mixup_batch(bf, fl, y, MIXUP_ALPHA)
        y = smooth(y, LABEL_SMOOTHING)
        with torch.amp.autocast("cuda", enabled=scaler is not None):
            logits = model(bf, fl)
            loss = criterion(logits, y)
        optimizer.zero_grad(set_to_none=True)
        if scaler is not None:
            scaler.scale(loss).backward()
            if GRAD_CLIP > 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            old_scale = scaler.get_scale()
            scaler.step(optimizer); scaler.update()
            if scaler.get_scale() >= old_scale: sched.step()
        else:
            loss.backward()
            if GRAD_CLIP > 0: nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step(); sched.step()
        losses.append(loss.item())
        ps.append(torch.sigmoid(logits).detach().float().cpu().numpy())
        if log_every and (i + 1) % log_every == 0:
            dt = time.time() - t_last; t_last = time.time()
            print(f"    step {i+1}/{len(loader)} | {dt:.1f}s | loss {float(np.mean(losses[-log_every:])):.4f}")
    hard_ys = np.concatenate(hard_ys); ps = np.concatenate(ps)
    auc = roc_auc_score(hard_ys, ps) if len(np.unique(hard_ys)) > 1 else float("nan")
    return float(np.mean(losses)), auc

# === Train ONE full-data model ===
print(f"\n=== Training full-data model ({EPOCHS} epochs, all {df_train['patient_id'].nunique()} patients) ===")
seed_everything(BASE_SEED + 100)
train_ds = CachedCellDataset(df_train, bf_train_cache, fl_train_cache,
                             train_modality_transform("bf"),
                             train_modality_transform("fl"),
                             paired_tf=PairedGeoAug())
sampler = PatientBalancedSampler(df_train, batch_size=BATCH_SIZE,
                                 patients_per_batch=PATIENTS_PER_BATCH, seed=BASE_SEED + 100)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)

model = MultimodalClassifier(pretrained=True, dropout=DROPOUT).to(DEVICE)
pos = (df_train["Diagnosis"] == 1).sum()
neg = (df_train["Diagnosis"] == 0).sum()
pos_weight = torch.tensor(neg / max(pos, 1), device=DEVICE)
print(f"  pos_weight={pos_weight.item():.3f}  LR={LR}  mixup={MIXUP_ALPHA}  ls={LABEL_SMOOTHING}")
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, steps_per_epoch=len(train_loader),
    epochs=EPOCHS, pct_start=0.1)
scaler = torch.amp.GradScaler("cuda") if DEVICE == "cuda" else None

history = []
for ep in range(EPOCHS):
    t0 = time.time()
    tr_loss, tr_auc = run_epoch_train(model, train_loader, optimizer, scaler, criterion, sched)
    dt = time.time() - t0
    print(f"  ep {ep:>2d} | tr_loss {tr_loss:.4f} tr_auc {tr_auc:.4f} | {dt:.1f}s")
    history.append({"epoch": ep, "tr_loss": tr_loss, "tr_auc": tr_auc, "time": dt})

ckpt_path = OUT_DIR / "fulldata_best.pt"
torch.save({"model": model.state_dict(), "epoch": EPOCHS - 1,
            "args": {"dropout": DROPOUT, "backbone": "efficientnet_b0" if USE_EFFICIENTNET else "resnet18"}},
           ckpt_path)
with open(OUT_DIR / "history.json", "w") as f:
    json.dump({"history": history}, f, indent=2)
print(f"\nSaved {ckpt_path}")
del model, optimizer, sched, scaler, train_loader
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
epochs = [e["epoch"] for e in history]
ax[0].plot(epochs, [e["tr_loss"] for e in history], marker="o", color="tab:blue")
ax[1].plot(epochs, [e["tr_auc"]  for e in history], marker="o", color="tab:green")
ax[0].set(title="Train BCE loss", xlabel="epoch", ylabel="loss")
ax[1].set(title="Train AUC",      xlabel="epoch", ylabel="AUC")
for a in ax: a.grid(True)
plt.tight_layout()
plt.savefig("/kaggle/working/learning_curves.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# === AdaBN: update BN running stats on test data before inference ===
@torch.no_grad()
def adabn_pass(model, loader):
    model.train()
    for batch in loader:
        bf = batch["bf"].to(DEVICE, non_blocking=True)
        fl = batch["fl"].to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=DEVICE == "cuda"):
            _ = model(bf, fl)
    model.eval()

def _d4_at_scale(bf, fl, scale=None):
    if scale is not None and scale != bf.shape[-1]:
        bf = F.interpolate(bf, size=(scale, scale), mode="bilinear", align_corners=False)
        fl = F.interpolate(fl, size=(scale, scale), mode="bilinear", align_corners=False)
    for k in range(4):
        bfr = torch.rot90(bf, k, dims=(-2, -1))
        flr = torch.rot90(fl, k, dims=(-2, -1))
        yield bfr, flr
        yield TF.hflip(bfr), TF.hflip(flr)

def _multiscale_tta(bf, fl, scales):
    for s in scales:
        for bf_t, fl_t in _d4_at_scale(bf, fl, scale=s):
            yield bf_t, fl_t

def load_model(ckpt_path):
    state = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    args = state.get("args", {})
    dropout = float(args.get("dropout", DROPOUT))
    model = MultimodalClassifier(pretrained=False, dropout=dropout).to(DEVICE)
    model.load_state_dict(state["model"]); model.eval()
    return model

def predict(ckpt_path, loader, tta_scales=None):
    model = load_model(ckpt_path)
    if USE_ADABN:
        print("  running AdaBN pass...")
        adabn_pass(model, loader)
    aug_fn = (lambda bf, fl: _multiscale_tta(bf, fl, tta_scales)) if tta_scales \
             else (lambda bf, fl: _d4_at_scale(bf, fl))
    n_aug = 8 * (len(tta_scales) if tta_scales else 1)
    preds = []
    with torch.no_grad():
        for batch in loader:
            bf = batch["bf"].to(DEVICE, non_blocking=True)
            fl = batch["fl"].to(DEVICE, non_blocking=True)
            p = None
            for bf_t, fl_t in aug_fn(bf, fl):
                with torch.amp.autocast("cuda", enabled=DEVICE == "cuda"):
                    pi = torch.sigmoid(model(bf_t, fl_t)).float()
                p = pi if p is None else p + pi
            preds.append((p / n_aug).cpu().numpy())
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return np.concatenate(preds)

test_ds = CachedCellDataset(df_test, bf_test_cache, fl_test_cache,
                            eval_modality_transform("bf"),
                            eval_modality_transform("fl"))
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

scales_to_use = TTA_SCALES if USE_MULTISCALE_TTA else None
n_aug_total = 8 * (len(TTA_SCALES) if USE_MULTISCALE_TTA else 1)
print(f"Predicting with {n_aug_total}-way TTA "
      f"(scales={scales_to_use or 'native'}, AdaBN={USE_ADABN})")

t0 = time.time()
preds = predict(ckpt_path, test_loader, tta_scales=scales_to_use)
print(f"  done in {time.time()-t0:.1f}s")

sub = pd.DataFrame({"Name": df_test["Name"].values, "Diagnosis": preds})
sub.to_csv("/kaggle/working/submission.csv", index=False)
print(f"\nWrote submission.csv  (mean pred = {preds.mean():.3f}, "
      f"min {preds.min():.3f}, max {preds.max():.3f})")
print(sub.head())
!wc -l /kaggle/working/submission.csv